# 세그먼트 x 프로모션 SHAP + Permutation Importance 분석

**3개 scope(overall / promotion_only / nonpromotion_only) x 6개 세그먼트(A1~A3, B1~B3)
= 18개 그룹**에 대해 각각 SHAP / Permutation Importance를 계산한다.

계산 결과는 파일로 저장하지 않고 **노트북 화면에 표/그래프로만 출력**한다.

그룹당 모델은 **한 번만 학습**해서 SHAP과 Permutation Importance 양쪽에 재사용한다
(따로 학습하면 18 그룹 x 2 = 36번 학습이 되어 두 배로 느려진다).

## 그룹 구성 및 모델 매칭 로직
세그먼트는 프로모션 여부가 아니라 시청 행동(w1+w2, w3) 기준으로 나뉘어 있어,
세그먼트 자체에는 프로모션·비프로모션 유저가 섞여 있다. 그래서 그룹마다
**그 그룹의 프로모션 구성에 맞는 모델**을 매칭한다.

| 그룹 | 대상 데이터 | 사용 모델 | `is_promotion` 피처 |
|---|---|---|---|
| `overall__A1` ~ `overall__B3` (6개) | 해당 세그먼트 전체 (프로모션 혼합) | `tuned_model_overall` | 포함 (값이 섞여 있어 변별력 있음) |
| `promotion_only__A1` ~ `__B3` (6개) | 해당 세그먼트 중 `is_promotion==1` | `tuned_model_promotion_only` | 제외 (그룹 내 상수) |
| `nonpromotion_only__A1` ~ `__B3` (6개) | 해당 세그먼트 중 `is_promotion==0` | `tuned_model_nonpromotion_only` | 제외 (그룹 내 상수) |


In [ ]:
# Cell 1: 라이브러리
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import shap
import warnings

warnings.filterwarnings('ignore')
print('라이브러리 로드 완료')


In [ ]:
# Cell 2: 데이터 + 세그먼트 + 튜닝 모델 로드
RANDOM_STATE = 42
CACHE_DIR = Path('cache')

exp_df = pd.read_csv(CACHE_DIR / 'expanded_dataset.csv')
seg_df = pd.read_csv(CACHE_DIR / 'step07_segment_assignment.csv')[['USER_KEY', 'segment']]

df = exp_df.merge(seg_df, on='USER_KEY', how='inner')
print(f'merged: {df.shape}')
print(df['segment'].value_counts().sort_index())
print()
print('세그먼트 내 프로모션 구성 (행: segment, 열: is_promotion):')
print(pd.crosstab(df['segment'], df['is_promotion']))

MODELS = {
    'overall':           joblib.load(CACHE_DIR / 'tuned_model_overall.pkl'),
    'promotion_only':    joblib.load(CACHE_DIR / 'tuned_model_promotion_only.pkl'),
    'nonpromotion_only': joblib.load(CACHE_DIR / 'tuned_model_nonpromotion_only.pkl'),
}
print()
print('튜닝 모델 로드 완료:', list(MODELS.keys()))


In [ ]:
# Cell 3: 피처 패밀리 분류 (06_XAI/step06.py 기준과 동일하게 맞춤)
PAYMENT_FEATURES = ['payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios']

FEATURE_FAMILY = {
    'usage_retention_behavior': [
        'watch_time', 'watch_session', 'retention', 'diff_between',
        'only_w', 'cold_start', 'recency', 'gap', 'inactive',
        'active_ratio', 'watch_per_day', 'rewatch', 'weekend',
    ],
    'content_preference': [
        'drama', 'comedy', 'romance', 'thriller', 'sf', 'horror',
        'action', 'family', 'documentary', 'historical', 'other',
        'movie', 'release', 'genre',
    ],
    'membership_context': [
        'is_standard', 'is_premium', 'is_basic',
        'is_churn_prevented', 'is_user_verified',
        'reg_is_weekend', 'reg_hour',
        'age_group', 'is_female', 'is_male',
    ],
    'acquisition_split': ['is_promotion'],
    'payment_proxy': PAYMENT_FEATURES,
}

def _family_of(feature):
    fn = feature.lower()
    for fam, keywords in FEATURE_FAMILY.items():
        if any(kw in fn for kw in keywords):
            return fam
    return 'other'

print('피처 패밀리 정의 완료')


In [ ]:
# Cell 4: SHAP / Permutation Importance 함수
# 두 함수 모두 '이미 학습된 모델'을 받는다 (그룹당 학습은 1번만 -> Cell 6에서 수행)
def run_shap(df_scope, features, fitted_model, sample_max=5000):
    X = df_scope[features].apply(pd.to_numeric, errors='coerce').fillna(0)

    if len(X) > sample_max:
        idx = np.random.RandomState(RANDOM_STATE).choice(len(X), sample_max, replace=False)
        X_sample = X.iloc[idx]
    else:
        X_sample = X

    explainer = shap.TreeExplainer(fitted_model)
    vals = explainer.shap_values(X_sample)
    if isinstance(vals, list):
        vals = vals[1] if len(vals) > 1 else vals[0]
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[:, :, 1] if vals.shape[2] > 1 else vals[:, :, 0]

    mean_abs = np.abs(vals).mean(axis=0)
    importance = pd.DataFrame({
        'feature': features,
        'mean_abs_shap': mean_abs,
        'family': [_family_of(f) for f in features],
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    return importance


def run_permutation_importance(df_scope, features, fitted_model, n_repeats=5):
    X = df_scope[features].apply(pd.to_numeric, errors='coerce').fillna(0)
    y = (1 - df_scope['is_repurchase'].astype(int)).to_numpy()

    def _auc(est, X_, y_):
        return roc_auc_score(y_, est.predict_proba(X_)[:, 1])

    result = permutation_importance(
        fitted_model, X, y,
        scoring=_auc, n_repeats=n_repeats,
        random_state=RANDOM_STATE, n_jobs=1,
    )

    importance = pd.DataFrame({
        'feature': features,
        'perm_importance': result.importances_mean,
        'perm_std': result.importances_std,
        'family': [_family_of(f) for f in features],
    }).sort_values('perm_importance', ascending=False).reset_index(drop=True)

    return importance

print('SHAP / Permutation Importance 함수 정의 완료')


In [ ]:
# Cell 5: 18개 분석 그룹 정의 (3 scope x 6 segment)
SEGMENTS = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3']
SCOPES = ['overall', 'promotion_only', 'nonpromotion_only']
EXCLUDE_BASE = {'USER_KEY', 'is_repurchase', 'segment'}

GROUPS = {}
for seg in SEGMENTS:
    seg_pop = df[df['segment'] == seg]

    # (1) 세그먼트 전체 (프로모션 혼합) -> overall 모델, is_promotion 포함
    GROUPS[f'overall__{seg}'] = (seg_pop.copy(), MODELS['overall'], True)

    # (2) 세그먼트 중 프로모션 가입자 -> promotion_only 모델, is_promotion 제외(상수)
    GROUPS[f'promotion_only__{seg}'] = (
        seg_pop[seg_pop['is_promotion'] == 1].copy(), MODELS['promotion_only'], False
    )

    # (3) 세그먼트 중 비프로모션 가입자 -> nonpromotion_only 모델, is_promotion 제외(상수)
    GROUPS[f'nonpromotion_only__{seg}'] = (
        seg_pop[seg_pop['is_promotion'] == 0].copy(), MODELS['nonpromotion_only'], False
    )

print(f'총 {len(GROUPS)}개 그룹 (3 scope x {len(SEGMENTS)} segment)')
for name, (g, _, inc_promo) in GROUPS.items():
    print(f'{name:<28} rows={len(g):>6}  include_is_promotion={inc_promo}')


In [ ]:
# Cell 6: 그룹별 모델 학습(1회) -> SHAP + Permutation Importance 계산
# 학습은 그룹당 1번만 수행하고(총 18번), 같은 모델을 SHAP/Permutation 양쪽에 재사용한다.
RESULTS = {}

for name, (df_grp, base_model, inc_promo) in GROUPS.items():
    exclude = EXCLUDE_BASE | (set() if inc_promo else {'is_promotion'})
    features = [c for c in df.columns if c not in exclude]

    X = df_grp[features].apply(pd.to_numeric, errors='coerce').fillna(0)
    y = (1 - df_grp['is_repurchase'].astype(int)).to_numpy()

    print(f'[{name}] rows={len(df_grp)}  features={len(features)}  학습 중...')
    fitted_model = clone(base_model)
    fitted_model.fit(X, y)

    print(f'[{name}] SHAP 계산 중...')
    shap_imp = run_shap(df_grp, features, fitted_model)
    print(f'[{name}] Permutation Importance 계산 중...')
    perm_imp = run_permutation_importance(df_grp, features, fitted_model)

    RESULTS[name] = {'shap': shap_imp, 'permutation': perm_imp}

print()
print('전체 그룹 계산 완료:', len(RESULTS), '개')


In [ ]:
# Cell 7: 그룹별 SHAP / Permutation Importance Top 10 비교 표
TOP_N = 10

for name, res in RESULTS.items():
    print()
    print(f'=== {name}  (n={len(GROUPS[name][0])}) ===')
    shap_top = res['shap'].head(TOP_N)[['feature', 'mean_abs_shap', 'family']].reset_index(drop=True)
    perm_top = res['permutation'].head(TOP_N)[['feature', 'perm_importance', 'family']].reset_index(drop=True)
    side_by_side = pd.concat(
        [shap_top.add_prefix('SHAP_'), perm_top.add_prefix('PERM_')],
        axis=1,
    )
    display(side_by_side)


In [ ]:
# Cell 8: 세그먼트(행) x scope(열) 6x3 그리드로 SHAP Top5 비교 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(SEGMENTS), len(SCOPES), figsize=(18, 30))
for row, seg in enumerate(SEGMENTS):
    for col, scope in enumerate(SCOPES):
        name = f'{scope}__{seg}'
        ax = axes[row, col]
        top5 = RESULTS[name]['shap'].head(5).iloc[::-1]
        ax.barh(top5['feature'], top5['mean_abs_shap'], color='#4C72B0')
        ax.set_title(f'{name}  (n={len(GROUPS[name][0])})', fontsize=9)
        ax.set_xlabel('mean |SHAP value|', fontsize=8)
        ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.show()
